# multi_1h_jma_atr_004

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (12).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1h` |
| Code cells | 10 |
| Detected functions | resample_1hr_custom, __init__, compute_atr, compute_jma_and_signals, compute_atr_max, cy_backtest_trades, backtest_trades |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_parquet('/content/drive/MyDrive/Binance_Crypto_T5_2025.parquet')
data.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data

In [ ]:
data['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data['Date'] = pd.to_datetime(data['Date'])

# Define the date range
start_date = '2025-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['BTCUSDT']  # Add tickers you want

# Apply filtering
data = data[
    (data["Date"] >= start_date) &
    (data["Date"] <= end_date) &
    (data["Ticker"].isin(selected_tickers))
]

# Display filtered results
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('1h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close', 'volume']]

data = data_1hr
data

### **INDICATOR CODE**

In [ ]:
import pandas as pd
import numpy as np

# ===============================
# ATR Calculator
# ===============================
class ATRIndicator:
    """Average True Range (ATR) Calculator"""
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high = df[self.high_col]
        low = df[self.low_col]
        close = df[self.close_col]

        tr = pd.concat([
            high - low,
            (high - close.shift(1)).abs(),
            (low - close.shift(1)).abs()
        ], axis=1).max(axis=1)

        atr = tr.rolling(window=period, min_periods=1).mean()
        return atr.shift(shift)  # shift to prevent lookahead

# ===============================
# JMA + Signals (manual loop)
# ===============================
def compute_jma_and_signals(df,
                            smooth_length=14,
                            normalization_period=36,
                            upper_threshold=1.5,
                            lower_threshold=-1.0,
                            rs_period=14,
                            rs_percentile=10):
    df = df.copy()
    price = df['close']

    # --- JMA calculation (manual loop) ---
    alpha = 2.0 / (smooth_length + 1)
    jma_values = [price.iloc[0]]
    for p in price.iloc[1:]:
        jma_values.append(jma_values[-1] + (p - jma_values[-1]) * alpha)
    df['JMA'] = jma_values

    # --- Normalization ---
    df['Normalized_JMA'] = (df['JMA'] - df['JMA'].rolling(normalization_period).mean()) / \
                            df['JMA'].rolling(normalization_period).std()

    # --- Rogers-Satchell Volatility ---
    o = df['open'].replace(0, np.nan)
    h = df['high']
    l = df['low']
    c = df['close'].replace(0, np.nan)

    log_hc = np.log(h / c)
    log_ho = np.log(h / o)
    log_lc = np.log(l / c)
    log_lo = np.log(l / o)

    df['RS_Volatility'] = ((log_hc * log_ho + log_lc * log_lo).rolling(window=rs_period).mean()) * 1e6

    # --- RS Volatility Filter ---
    df['RS_Threshold'] = df['RS_Volatility'].rolling(normalization_period).quantile(rs_percentile / 100.0)

    # --- Signals ---
    df['Signal'] = 0
    df.loc[(df['Normalized_JMA'] > upper_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 1
    df.loc[(df['Normalized_JMA'] < lower_threshold) & (df['RS_Volatility'] > df['RS_Threshold']), 'Signal'] = 2

    return df

# ===============================
# ATR_max computation helper
# ===============================
def compute_atr_max(group):
    return pd.concat([
        group['ATR_14'].rolling(20).max() * 0.5,
        group['ATR_5'].rolling(20).max() * 0.5
    ], axis=1).max(axis=1)

# ===============================
# Full Data Preparation
# ===============================
# Assuming 'data' DataFrame exists with required columns
data['Date'] = pd.to_datetime(data['Date'])

# Apply JMA + Signals per ticker
jma_results = []
for ticker, group in data.groupby('Ticker'):
    jma_group = compute_jma_and_signals(group)
    jma_results.append(jma_group)
data = pd.concat(jma_results, ignore_index=True)

# Initialize ATR calculator
atr_calc = ATRIndicator()

# Compute ATRs per ticker with shift=1
for period in [14, 5]:
    col_name = f'ATR_{period}'
    atr_results = []
    for ticker, group in data.groupby('Ticker'):
        atr_vals = atr_calc.compute_atr(group, period=period, shift=1)
        atr_results.append(atr_vals)
    data[col_name] = pd.concat(atr_results, ignore_index=True)

# Compute ATR_max
atr_max_results = []
for ticker, group in data.groupby('Ticker'):
    atr_max_vals = compute_atr_max(group)
    atr_max_results.append(atr_max_vals)
data['ATR_max'] = pd.concat(atr_max_results, ignore_index=True)

# Shift signals for next-bar execution
data['Signal'] = data.groupby('Ticker', group_keys=False)['Signal'].shift(1)
data = data.dropna(subset=['Signal'])
data['Signal'] = data['Signal'].astype(np.int8)

# Ready for backtesting
data

In [ ]:
load_ext cython

# **BACKTESTING CODE**

In [ ]:
%%cython
import numpy as np
cimport numpy as np

def cy_backtest_trades(
    object[:, :] data,
    str ticker,
    double capital,
    double fee_rate,
    double slippage_rate,
    double tp_pct,
    double sl_pct,
    double entry_sl_pct
):
    cdef:
        int i, n = data.shape[0]
        bint position_open = False
        bint exit_next_open = False

        # Trade State
        double entry_price = 0.0, quantity = 0.0
        object entry_time = None
        int direction = 0  # 1 = long, -1 = short
        double tp_price = 0.0, sl_price = 0.0, esl_price = 0.0
        object last_exit_time = None

        # Output
        list trades = []
        double cumulative_pnl = 0.0

        # Candle data
        object time
        double open_, high, low, close
        int signal

        # Exit
        double exit_price, pnl, fee, net_pnl
        str reason

    for i in range(n):
        time = data[i, 0]
        open_ = <double>data[i, 1]
        high = <double>data[i, 2]
        low = <double>data[i, 3]
        close = <double>data[i, 4]
        signal = <int>data[i, 5]

        # === Exit on Open (delayed SL confirmation) ===
        if exit_next_open:
            exit_price = open_ * (1 + slippage_rate) if direction == -1 else open_ * (1 - slippage_rate)
            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl

            trades.append((
                ticker, entry_time, time, entry_price, exit_price, net_pnl,
                "SL Close Confirmed", cumulative_pnl,
                "long" if direction == 1 else "short"
            ))

            # Reset
            position_open = False
            exit_next_open = False
            last_exit_time = time
            continue

        # === Entry Logic ===
        if not position_open and time != last_exit_time and signal in (1, 2):
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate) if direction == 1 else open_ * (1 - slippage_rate)
            quantity = capital / entry_price
            entry_time = time
            position_open = True

            # Initialize SL/TP
            sl_price = entry_price * (1 - sl_pct * direction)
            tp_price = entry_price * (1 + tp_pct * direction)
            esl_price = entry_price * (1 - entry_sl_pct * direction)

            # === Entry SL Check ===
            if (direction == 1 and low <= esl_price) or (direction == -1 and high >= esl_price):
                exit_price = esl_price
                pnl = (exit_price - entry_price) * quantity * direction
                fee = fee_rate * (entry_price + exit_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl

                trades.append((
                    ticker, entry_time, time, entry_price, exit_price, net_pnl,
                    "Entry SL", cumulative_pnl,
                    "long" if direction == 1 else "short"
                ))

                position_open = False
                last_exit_time = time
                continue

            # === Delayed SL Exit Check ===
            if (direction == 1 and close <= sl_price) or (direction == -1 and close >= sl_price):
                exit_next_open = True

            continue  # Skip to next candle

        # === Position Management ===
        if position_open:
            # --- Trailing SL Update ---
            if direction == 1:
                sl_price = max(sl_price, high * (1 - sl_pct))
            else:
                sl_price = min(sl_price, low * (1 + sl_pct))

            # --- Trailing TP Update ---
            if direction == 1:
                tp_price = max(tp_price, high * (1 - tp_pct))
            else:
                tp_price = min(tp_price, low * (1 + tp_pct))

            # --- Exit Checks ---
            if (direction == 1 and high >= tp_price) or (direction == -1 and low <= tp_price):
                exit_price = tp_price
                reason = "Trailing TP"
            elif (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price):
                exit_price = sl_price
                reason = "Stop Loss"
            else:
                continue  # Still holding

            # --- Exit Execution ---
            pnl = (exit_price - entry_price) * quantity * direction
            fee = fee_rate * (entry_price + exit_price) * quantity
            net_pnl = pnl - fee
            cumulative_pnl += net_pnl

            trades.append((
                ticker, entry_time, time, entry_price, exit_price, net_pnl,
                reason, cumulative_pnl,
                "long" if direction == 1 else "short"
            ))

            position_open = False
            last_exit_time = time

    return trades

In [ ]:
from tqdm import tqdm
import numpy as np
import pandas as pd

def backtest_trades(
    data: pd.DataFrame,
    tp_pct: float,
    sl_pct: float,
    entry_sl_pct: float
) -> pd.DataFrame:

    initial_capital = 100000
    fee_rate = 0.00015  # 0.04% round trip
    slippage_rate = 0.0002  # 0.05% round trip

    # Convert 'Date' to datetime and avoid deprecated warning
    data['Date'] = pd.to_datetime(data['Date'])
    data['Date'] = np.array(data['Date'].dt.to_pydatetime())

    tickers = data['Ticker'].unique()

    # Prepare each ticker's data for cython backtest
    ticker_map = {
        ticker: data[data['Ticker'] == ticker][[
            "Date", "open", "high", "low", "close", "Signal"
        ]].to_numpy(dtype=object)
        for ticker in tickers
    }

    all_trades = []

    for ticker in tqdm(tickers, desc="Backtesting"):
        trades = cy_backtest_trades(
            ticker_map[ticker],
            ticker,
            initial_capital,
            fee_rate,
            slippage_rate,
            tp_pct,
            sl_pct,
            entry_sl_pct  # Used for Entry SL check
        )
        all_trades.extend(trades)

    return pd.DataFrame(all_trades, columns=[
        "Ticker", "Entry Time", "Exit Time", "Entry Price", "Exit Price",
        "PnL", "Exit Reason", "Cumulative PnL", "Direction"
    ])

tradebook = backtest_trades(
    data,
    tp_pct=0.1,            # 6% TP
    sl_pct=0.001,           # 0.1% SL
    entry_sl_pct=0.03,      # 2% Entry SL
)

tradebook.to_csv("/content/drive/MyDrive/tradebook_BTCUSDT_2022_25.csv", index=False)
tradebook
